# AI-Powered Indie & Mellow Music Engine
**Fokus Keahlian:** Data Science & Machine Learning

**Latar Belakang Riset:**
Di era pascamodern, musik telah bertransformasi menjadi rutinitas harian yang esensial. Bagi Generasi Z, musisi *indie pop* dan balada naratif seperti Hindia, Bernadya, hingga Pamungkas telah menjadi primadona yang mendominasi preferensi audiens. **INDIE POP ADALAH KESUKAAN GENERASI Z DAN AKAN JADI YANG UTAMA YANG SAYA RISET**

Namun, di balik kemudahan algoritma *streaming* saat ini, sering kali muncul sebuah kendala teknis: *bagaimana jika fitur auto-playlist gagal menangkap nuansa spesifik yang kita inginkan, atau bahkan dihapus?* Riset independen ini lahir dari keresahan personal penulis saat kesulitan mengkurasi lagu-lagu berirama *mellow* yang akurat secara manual. 

Sebagai solusi, penulis membangun prototipe mesin rekomendasi (*Recommendation Engine*) menggunakan hibrida *Machine Learning* dan *Deep Learning* untuk membedah kedalaman lirik dan instrumen, menciptakan kurasi musik yang presisi tanpa bergantung pada algoritma bawaan platform.

In [2]:
!pip install kagglehub


   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   -------------------- ------------------- 1/2 [kagglehub]
   ---------------------------------------- 2/2 [kagglehub]



In [ ]:
import pandas as pd
import os

import kagglehub
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")


# 1. Memuat dataset Spotify
print("Sedang memuat dataset spotify data asli...")
df_mentah = pd.read_csv(os.path.join(path,"songs_with_attributes_and_lyrics.csv"))

# 2. Menginspeksi wujud asli data sebelum direduksi
print("\n--- ARSITEKTUR DATA MENTAH ---")
print(f"Dimensi (Baris, Kolom): {df_mentah.shape}")
print("\nDaftar 17 Kolom Asli:")
for i, kolom in enumerate(df_mentah.columns, 1):
    print(f"{i}. {kolom}")

 48%|████▊     | 429M/894M [00:23<00:28, 17.4MB/s] 

### Kamus Data (Data Dictionary) Spotify API
Dataset ini terdiri dari 17 variabel yang terbagi menjadi tiga kategori utama: Metadata Identitas, Fitur Analisis Audio (*Audio Features*), dan Data Tekstual.

| Kategori | Nama Kolom | Deskripsi |
| :--- | :--- | :--- |
| **Metadata** | `id` | ID alfanumerik unik Spotify untuk trek lagu. |
| | `name` | Judul lagu. |
| | `album_name` | Nama album dari lagu tersebut. |
| | `artists` | Nama musisi atau grup band. |
| | `duration_ms` | Durasi trek dalam satuan milidetik. |
| **Data Teks** | `lyrics` | Teks lirik lagu utuh hasil ekstraksi. |
| **Fitur Audio** | `valence` | Skor (0.0 - 1.0) tingkat emosi positif (ceria/bahagia). Semakin mendekati 0, semakin *mellow*/sedih. |
| | `acousticness` | Skor (0.0 - 1.0) tingkat keyakinan bahwa trek tersebut murni instrumen akustik tanpa *synthesizer*. |
| | `energy` | Skor (0.0 - 1.0) intensitas audio. Energi rendah berarti lagu bertempo lambat dan tenang. |
| | `danceability` | Skor (0.0 - 1.0) kecocokan irama dan ketukan lagu untuk digunakan menari. |
| | `instrumentalness` | Prediksi (0.0 - 1.0) ketiadaan vokal. Skor tinggi berarti lagu murni instrumen. |
| | `speechiness` | Deteksi vokal lisan (seperti *podcast*, *rap*, atau *spoken word*). |
| | `liveness` | Deteksi apakah lagu direkam secara *live* bersama penonton atau di dalam studio. |
| | `tempo` | Kecepatan trek dalam ukuran *Beats Per Minute* (BPM). |
| | `loudness` | Rata-rata tingkat volume trek dalam desibel (dB). |
| | `key` | Kunci nada dasar trek (menggunakan notasi standar *Pitch Class*). |
| | `mode` | Modalitas melodi trek (Mayor = 1, Minor = 0). |

*Catatan: Pemahaman mendalam terhadap kamus data ini menjadi landasan logis bagi proses Reduksi Dimensi dan Seleksi Fitur pada tahap selanjutnya.*

###  Seleksi Fitur (Feature Selection) & Kutukan Dimensi
Berdasarkan hasil distribusi di atas, kita mengunci empat metrik utama dan membuang sisanya untuk menghindari *Curse of Dimensionality* (*noise* yang membuat AI kebingungan):
*   **`valence`:** (Skor Emosi): 0 ke 1 (tambah naik tambah gak sedih), kita butuh buat tau emosinya naik atau turun sesuai genre

*   **`acousticness`:** (Skor Instrumen): 0 ke 1 (tambah naik tambah akustik heavy),kita butuh buat tau instumennya

*   **`energy`:** (Intensitas Audio): o ke 1 (tambah naik tambah berisik dan berisi kayak rock), kita butuh buat tau kualitasnya atau energinya

*   **`danceability`:** (Skor "Dansa"): 0 ke 1 (tambah naik tambah jedag jedug)



In [3]:
# 1. Reduksi 8 Kolom Target & Filter Baris Tanpa Lirik
kolom_target = ['id', 'name', 'artists', 'lyrics', 'valence', 'acousticness', 'energy', 'danceability']
df_valid = df_mentah[kolom_target].dropna(subset=['lyrics'])

# 2. DATA ENGINEERING: Golden Filter (Artis Indie & Diksi Eksklusif)
print("Mengekstrak mahakarya musisi indie lokal dan lirik puitis...")

# Memancing mesin langsung dengan kiblat musisi indie naratif 
target_artis = 'hindia|sal priadi|feby putri|pamungkas|bernadya|ardhito|danilla|kunto aji|banda neira|payung teduh|fourtwnty'

# Menggunakan Regex batas kata (\b) dan diksi eksklusif yang mustahil ada di bahasa Hindi
target_kata = r'\b(mengapa|kembali|seperti|semua|menjadi|pernah|waktu|cinta|rindu|fana|sendu)\b'

# Menyaring data: Ambil jika artisnya cocok ATAU liriknya mengandung diksi puitis Indonesia
kondisi_artis = df_valid['artists'].str.contains(target_artis, case=False, na=False)
kondisi_lirik = df_valid['lyrics'].str.contains(target_kata, case=False, na=False, regex=True)

df_filter_indo = df_valid[kondisi_artis | kondisi_lirik]
print(f"Total lagu lokal & indie yang berhasil dijaring: {df_filter_indo.shape[0]} lagu.")

# 3. Mengambil sampel acak 
jumlah_sampel = min(100000, df_filter_indo.shape[0])
df_clean = df_filter_indo.sample(n=jumlah_sampel, random_state=42).copy()
df_clean.reset_index(drop=True, inplace=True)

display(df_clean[['name', 'artists']].head(3))

Mengekstrak mahakarya musisi indie lokal dan lirik puitis...


C:\Users\asus\AppData\Local\Temp\ipykernel_19068\2389840586.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kondisi_lirik = df_valid['lyrics'].str.contains(target_kata, case=False, na=False, regex=True)


Total lagu lokal & indie yang berhasil dijaring: 1602 lagu.


,name,artists
0,Hitam Putih Dunia,Rocket Rockers
1,Pulang,Insomniacks
2,Biar Akhirnya Di Sini (feat. Henry Lamir),Dayang Nurfaizah


In [4]:
# 1. Mengaudit tipe data dan memastikan tidak ada nilai kosong (Null)
print("--- PROFIL STRUKTUR DATA ---")
df_clean.info()

# 2. Membedah distribusi statistik 4 metrik esensial
print("\n--- DISTRIBUSI STATISTIK FITUR AUDIO ---")
fitur_audio_stats = df_clean[['valence', 'acousticness', 'energy', 'danceability']]
display(fitur_audio_stats.describe())

--- PROFIL STRUKTUR DATA ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1602 entries, 0 to 1601
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            1602 non-null   object 
 1   name          1602 non-null   object 
 2   artists       1602 non-null   object 
 3   lyrics        1602 non-null   object 
 4   valence       1602 non-null   float64
 5   acousticness  1602 non-null   float64
 6   energy        1602 non-null   float64
 7   danceability  1602 non-null   float64
dtypes: float64(4), object(4)
memory usage: 100.3+ KB

--- DISTRIBUSI STATISTIK FITUR AUDIO ---


,valence,acousticness,energy,danceability
count,1602.000000,1602.000000,1602.000000,1602.000000
mean,0.486206,0.338819,0.622667,0.555479
std,0.247201,0.309380,0.217691,0.150108
min,0.023000,0.000000,0.010400,0.085600
25%,0.287000,0.045375,0.466250,0.462000
50%,0.446500,0.260000,0.630000,0.554000
75%,0.688000,0.596500,0.806000,0.661000
max,0.979000,0.996000,0.997000,0.956000


## Unsupervised Learning - Penggunaan K-Means
Karena dataset lagu ini tidak memiliki label kategori emosi yang eksplisit (seperti "Galau" atau "Ceria"), kita menggunakan pendekatan *Unsupervised Machine Learning*. yang labelnya tidak perlu dan ini cocok dengan kriteria datset

Algoritma *K-Means* akan mengukur jarak matematis dari keempat pilar metrik audio (valence, acousticness, energy, danceability), lalu membiarkan mesin mengelompokkan 20.000 lagu tersebut Berdasarkan intuisi eksplorasi data (EDA) kita, dunia musik secara garis besar bisa dibagi menjadi 5 spektrum emosi utama (Rock/Bising, EDM/Dansa, Pop/Galau, Akustik Ceria, dan Balada/Mellow). K-Means tidak tahu nama genrenya, ia hanya mencari 5 titik kumpul yang paling masuk akal secara matematis. Algoritma K-Means memulai kerjanya dengan menebak dan melempar 5 titik tengah (centroid) secara acak ke dalam lautan data dan kita butuh random_state agar tidak berubah ubah, lalu di uji 10 kali (standar)

### Standarisasi Skala & Pelatihan Model
Sebelum model dilatih, *Feature Scaling* wajib dilakukan. Algoritma berbasis jarak sangat sensitif terhadap skala angka. Jika tidak distandarisasi, fitur dengan rentang varians yang sedikit lebih lebar akan mendominasi pembentukan klaster dan membuat AI menjadi bias. dia hanya tau **angka besar itu besar dan kecil itu kecil makanya butuh standarscaller**

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. STANDARISASI DATA (Feature Scaling)
# Memisahkan 4 fitur metrik emosi untuk diolah secara matematis.
fitur_audio = df_clean[['valence', 'acousticness', 'energy', 'danceability']]

# StandardScaler akan menekan dan meratakan distribusi angka ke dalam skala "MANUSIA"
# Tujuannya agar metrik 'valence' dan 'acousticness' memiliki bobot pengaruh yang seimbang (tidak ada yang mendominasi).
scaler = StandardScaler()
audio_scaled = scaler.fit_transform(fitur_audio)

print(" mengelompokkan lagu...")

# 2. PELATIHAN MODEL AI (Model Training)
# n_clusters=5: Kita memerintahkan AI untuk membagi lagu menjadi 5 grup nuansa.
# random_state=42: Mengunci titik mula algoritma agar hasil grup (0-4) konsisten setiap kali di-run.
# n_init=10: Mesin akan mencoba 10 titik pusat acak yang berbeda dan mengunci hasil yang paling optimal.
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)

# .fit_predict() adalah fungsi ganda: mesin belajar dari data (fit), lalu langsung menebak grupnya (predict).
# Label grup (angka 0, 1, 2, 3, atau 4) langsung disuntikkan ke kolom baru bernama 'klaster_vibes'.
df_clean['klaster_vibes'] = kmeans.fit_predict(audio_scaled)

# 3. ANALISIS TITIK TENGAH (Centroid Analysis)
# Menghitung nilai rata-rata (mean) dari keempat metrik untuk setiap grup.
# Tabel ini adalah "kompas" kita untuk mencari tahu grup mana yang berisi lagu indie/mellow idamanmu.
print("\nKarakteristik Rata-Rata Metrik untuk ke-5 Klaster:")
hasil_klaster = df_clean.groupby('klaster_vibes')[['valence', 'acousticness', 'energy', 'danceability']].mean()

# Menampilkan hasil kompas klaster
display(hasil_klaster)

 mengelompokkan lagu...

Karakteristik Rata-Rata Metrik untuk ke-5 Klaster:


,valence,acousticness,energy,danceability
klaster_vibes,,,,
0,0.604560,0.643658,0.505734,0.653051
1,0.261378,0.772565,0.334013,0.437796
2,0.466130,0.060774,0.860030,0.466081
3,0.779874,0.193782,0.746444,0.704966
4,0.294297,0.220941,0.556805,0.514952


* **`valence`:** (Skor Emosi): 0 ke 1 (tambah naik tambah gak sedih), kita butuh buat tau emosinya naik atau turun sesuai genre

*   **`acousticness`:** (Skor Instrumen): 0 ke 1 (tambah naik tambah akustik heavy),kita butuh buat tau instumennya

*   **`energy`:** (Intensitas Audio): o ke 1 (tambah naik tambah berisik dan berisi kayak rock), kita butuh buat tau kualitasnya atau energinya

*   **`danceability`:** (Skor "Dansa"): 0 ke 1 (tambah naik tambah jedag jedug)

# 5 kluster ini kita bedah

* **klaster 0** = punya emosi yang ceria (0.60) dan akustik tinggi (0.64) dengan energi menengah. Ini adalah lagu pop-akustik atau folk pop santai, yang mungkin kita (generasi Z) kenal seperti karya Ardhito Pramono atau Banda Neira.
* **klaster 1** = akustik instrument tertinggi (0.77), energi terendah (0.33) dan emosi juga rendah atau melankolis (0.26). Ini habitat mutlak lagu balada naratif yang romantis dan syahdu. Pastinya ini yang **dicari**.
* **klaster 2** = energi tertinggi (0.86) dan akustik yang hancur (0.06). Ini mewakili vibes rock, metal, pop-punk, atau alternatif bising.
* **klaster 3** = punya emosi paling ceria (0.77) dan ritme dansa tinggi (0.70). Ini adalah pop riang bertempo cepat atau musik EDM *jedag jedug*.
* **klaster 4** = emosinya sedih (0.29) tapi energinya lumayan bertenaga (0.55) tanpa instrumen murni (0.22). Ini adalah tipikal lagu pop galau patah hati versi *full band* (seperti D'MASIV atau Dewa 19).

###  Ekstraksi Target (Mellow Acoustic)
Karena Klaster 1 terbukti sebagai habitat asli lagu-lagu *mellow* dan akustik yang kita cari, langkah selanjutnya adalah mengisolasi klaster ini. Lagu-lagu dari klaster lain akan kita singkirkan dari memori karena tidak relevan dengan target emosi proyek ini.

In [12]:
# Mengisolasi dataset hanya untuk Klaster 4 (Mellow/Akustik)
df_mellow = df_clean[df_clean['klaster_vibes'] == 1].copy()
df_mellow.reset_index(drop=True, inplace=True)
print(f"Total lagu mellow hasil saringan AI: {df_mellow.shape[0]} lagu.")

Total lagu mellow hasil saringan AI: 269 lagu.


## Ekstraksi Bahasa Lokal & Prapemrosesan NLP
Dataset mentah Spotify memuat hampir 1 juta lagu dari seluruh dunia. Karena target mesin rekomendasi ini adalah musik *indie* dan balada naratif Indonesia dengan lirik yang romantis, kita harus memfilter data menggunakan pustaka `langdetect`.

**Optimasi Komputasi:** Deteksi bahasa dan pembersihan teks menggunakan *Regular Expression* (RegEx) sengaja dilakukan *setelah* K-Means membuang klaster lain. Strategi memangkas beban kerja RAM secara drastis, memastikan proses komputasi tetap ringan dan efisien. Lirik kemudian disterilisasi dari tanda baca dan penanda struktur lagu (seperti `[Chorus]`) agar AI fokus pada makna kata.

In [13]:
from langdetect import detect
import re

print("Mendeteksi lagu berbahasa Indonesia...")

# Fungsi filter bahasa
def deteksi_bahasa(teks):
    try:
        return detect(str(teks))
    except:
        return "unknown"

df_mellow['bahasa'] = df_mellow['lyrics'].apply(deteksi_bahasa)
df_indo = df_mellow[df_mellow['bahasa'] == 'id'].copy()
df_indo.reset_index(drop=True, inplace=True)

# Fungsi pembersih lirik dari noise
def bersihkan_teks(teks):
    teks = str(teks).lower() 
    teks = re.sub(r'\[.*?\]|\(.*?\)', '', teks) 
    teks = re.sub(r'[^a-z\s]', '', teks) 
    return teks

df_indo['lirik_bersih'] = df_indo['lyrics'].apply(bersihkan_teks)
print(f"Total Lagu Mellow Indonesia: {df_indo.shape[0]} lagu.")

Mendeteksi lagu berbahasa Indonesia...
Total Lagu Mellow Indonesia: 235 lagu.


## Mesin Rekomendasi NLP (TF-IDF & Cosine Similarity)
Komputer tidak memahami lirik puitis; ia hanya mengerti representasi numerik. Oleh karena itu, kita menggunakan algoritma **TF-IDF (Term Frequency-Inverse Document Frequency)** untuk mengonversi teks menjadi matriks matematika. 

Agar mesin fokus pada diksi *mellow* (seperti "luka", "fana", "sendu"), kita mendefinisikan *Stopwords* khusus bahasa Indonesia untuk mengabaikan kata hubung pasaran. Terakhir, metrik jarak **Cosine Similarity** (YANG MIRIP MIRIP) digunakan untuk mengukur sudut kedekatan makna antar lagu. Lagu dengan lirik yang paling mirip akan memiliki skor mendekati 1.0.

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. MEMPERBARUI MATRIKS AI (Sinkronisasi dengan 235 lagu terbaru)
stopword_indo = ['dan', 'di', 'ke', 'dari', 'yang', 'ini', 'itu', 'untuk', 'pada', 'dengan', 'adalah', 'aku', 'kamu', 'dia', 'mereka', 'kita', 'kami', 'yg', 'nya']

tfidf = TfidfVectorizer(max_features=5000, stop_words=stopword_indo)
matriks_indo = tfidf.fit_transform(df_indo['lirik_bersih'])
kemiripan_indo = cosine_similarity(matriks_indo)

# 2. MENDEFINISIKAN ULANG FUNGSI
def rekomendasikan_lagu_indo(index, jumlah=3):
    skor_urut = sorted(list(enumerate(kemiripan_indo[index])), key=lambda x: x[1], reverse=True)[1:jumlah+1]
    
    judul = df_indo.iloc[index]['name']
    artis = df_indo.iloc[index]['artists']
    print(f"Referensi lirik: '{judul}' oleh {artis}\n")
    print("AI merekomendasikan karya mellow ini:")
    
    for i, (idx, skor) in enumerate(skor_urut, 1):
        j = df_indo.iloc[idx]['name']
        a = df_indo.iloc[idx]['artists']
        print(f"{i}. {j} - {a} (Kemiripan Makna: {skor:.2f})")

# 3. PENCARIAN & EKSEKUSI DINAMIS
print("-" * 50)
target_musisi = "Yura Yunita"

pencarian = df_indo[df_indo['artists'].str.contains(target_musisi, case=False, na=False)]

if not pencarian.empty:
    index_ditemukan = pencarian.index[0] 
    rekomendasikan_lagu_indo(index=index_ditemukan)
else:
    print(f"Musisi '{target_musisi}' tidak terambil di sampel.")
    display(df_indo[['name', 'artists']].head(20))

--------------------------------------------------
Referensi lirik: 'Tutur Batin' oleh Yura Yunita

AI merekomendasikan karya mellow ini:
1. Sial - Mahalini (Kemiripan Makna: 0.19)
2. Tak Lelah - Anuar Zain (Kemiripan Makna: 0.16)
3. Jangan Cintai Aku Apa Adanya - Tulus (Kemiripan Makna: 0.16)
